# Step 5. Profile every collected file before any change

For each raw or collected file: its shape, the dtype of every column, the number of empty cells, an example value and what the column means. Nothing is changed here. The cleaning in notebooks 02 and 07 starts from these files.

In [1]:
# Setup: run every notebook from the project folder, so raw/, interim/, clean/ and data/ resolve the same way.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise SystemExit("README.md not found above " + str(Path.cwd()) + ". Open the notebook from inside the project folder.")
os.chdir(ROOT)
print("Project folder:", ROOT)

Project folder: /Users/kushalvurukonda/Downloads/dcv-project


## Files in raw/ (never edited)

In [2]:
for f in sorted(Path("raw").rglob("*")):
    if f.is_file() and "openalex" not in f.parts:
        print(f"{str(f):60s} {f.stat().st_size / 1024:9.1f} KB")
cache = Path("raw/openalex")
print("\ncached OpenAlex JSON answers:", len(list(cache.glob("*.json"))) if cache.exists() else 0)

raw/.DS_Store                                                      6.0 KB
raw/nirf_overall_2017_2026-10-31.html                            602.0 KB
raw/nirf_overall_2017_2026-11-02.html                            602.0 KB
raw/nirf_overall_2018_2026-10-31.html                            636.3 KB
raw/nirf_overall_2018_2026-11-02.html                            636.3 KB
raw/nirf_overall_2019_2026-10-31.html                            119.8 KB
raw/nirf_overall_2019_2026-11-02.html                            119.8 KB
raw/nirf_overall_2020_2026-10-31.html                             95.8 KB
raw/nirf_overall_2020_2026-11-02.html                             95.8 KB
raw/nirf_overall_2021_2026-10-31.html                            109.2 KB
raw/nirf_overall_2021_2026-11-02.html                            109.2 KB
raw/nirf_overall_2022_2026-10-31.html                            109.3 KB
raw/nirf_overall_2022_2026-11-02.html                            109.3 KB
raw/nirf_overall_2023_2026-10-31.html 

## Shape, dtypes and meaning of each file

In [3]:
import glob, pandas as pd
pd.set_option("display.width", 220, "display.max_colwidth", 90, "display.max_rows", 100)
MEANINGS = {'NIRF Overall pages, parsed (interim/nirf_overall_scraped_*.csv)': ('interim/nirf_overall_scraped_*.csv', {'year': 'NIRF ranking year', 'nirf_id': 'id NIRF prints for the institution that year (the 2017 format differs)', 'name': 'institution name as printed, in capitals', 'location': 'city as printed', 'rank_page': 'rank read from the rank cell', 'cell_3': 'table cell 3 as read (state), kept raw to check the columns', 'cell_4': 'table cell 4 as read (a score), kept raw to check the columns', 'cell_5': 'table cell 5 as read (the rank), kept raw to check the columns', 'tlr': 'Teaching, Learning and Resources score (0 to 100)', 'rpc': 'Research and Professional Practice score (0 to 100)', 'go': 'Graduation Outcomes score (0 to 100)', 'oi': 'Outreach and Inclusivity score (0 to 100)', 'pr': 'Perception score (0 to 100)', 'rank_order': 'position of the row on the page (1 = first)', 'rank': 'NIRF Overall rank used in the analysis', 'score': 'copy of rpc, the research score'}), 'OpenAlex institution records (interim/openalex_institutions_*.csv)': ('interim/openalex_institutions_*.csv', {'ror_id': 'ROR id used to look the institution up', 'openalex_id': 'OpenAlex institution id', 'display_name': 'name OpenAlex uses', 'type': 'institution type in OpenAlex', 'country_code': 'ISO country code', 'city': 'city in OpenAlex', 'works_count_total': 'all works ever, per OpenAlex', 'cited_by_count_total': 'all citations ever, per OpenAlex', 'h_index': "h-index of the institution's works", 'two_yr_mean_citedness': 'mean citations of works in the last two years', 'lineage': 'institution ids counted together (the institution and units beneath it)', 'n_lineage': 'how many ids are in the lineage', 'ror_returned_differs': 'True if OpenAlex returned a different ROR id than asked', 'own_total_over_works_count': 'own works / works_count_total, a coverage check', 'status': 'ok, or the reason the record is unusable', 'source_url': 'URL called', 'retrieved_on': 'date retrieved'}), 'OpenAlex yearly counts (interim/openalex_yearly_*.csv)': ('interim/openalex_yearly_*.csv', {'ror_id': 'ROR id', 'openalex_id': 'OpenAlex institution id', 'year': 'publication year', 'works_all_lineage': 'works that year, institution plus units beneath it', 'works_article_lineage': 'of those, type = article', 'works_cited_ge1_lineage': 'of those, with at least 1 citation', 'works_cited_ge10_lineage': 'of those, with at least 10 citations', 'works_all_own': 'works that year whose authors list the institution itself', 'oa_works_count_year': "OpenAlex's own yearly count (starts in 2010)", 'oa_cited_by_count_year': 'citations received that year, to all works (starts in 2010)', 'future_year': 'True if the year is after the collection date'}), 'ROR to OpenAlex crosswalk (interim/openalex_crosswalk_*.csv)': ('interim/openalex_crosswalk_*.csv', {'id_new': 'our institution id', 'name_new': 'our institution name', 'decision': 'how the ROR match was made: auto or manual', 'ror_id': 'ROR id', 'openalex_id': 'OpenAlex institution id'})}

def profile(title, pattern, meaning):
    files = sorted(glob.glob(pattern))
    print("=" * 100); print(title)
    if not files:
        print("  file not found yet, run the collection step first"); return
    df = pd.read_csv(files[-1])
    print("  file :", files[-1]); print("  shape:", df.shape)
    out = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "example": df.iloc[0].astype(str).str.slice(0, 40),
                        "meaning": pd.Series(meaning)})
    display(out)

for title, (pattern, meaning) in MEANINGS.items():
    profile(title, pattern, meaning)

NIRF Overall pages, parsed (interim/nirf_overall_scraped_*.csv)
  file : interim/nirf_overall_scraped_2026-11-02.csv
  shape: (901, 16)


,dtype,missing,example,meaning
year,int64,0,2017,NIRF ranking year
nirf_id,str,0,IR17-I-2-18243,id NIRF prints for the institution that year (the 2017 format differs)
name,str,0,Indian Institute of Science Bangalore,"institution name as printed, in capitals"
location,str,0,Bengaluru,city as printed
rank_page,int64,0,1,rank read from the rank cell
cell_3,str,0,Karnataka,"table cell 3 as read (state), kept raw to check the columns"
cell_4,float64,0,83.28,"table cell 4 as read (a score), kept raw to check the columns"
cell_5,int64,0,1,"table cell 5 as read (the rank), kept raw to check the columns"
tlr,float64,0,83.11,"Teaching, Learning and Resources score (0 to 100)"
rpc,float64,0,87.59,Research and Professional Practice score (0 to 100)


OpenAlex institution records (interim/openalex_institutions_*.csv)
  file : interim/openalex_institutions_2026-11-02.csv
  shape: (164, 17)


,dtype,missing,example,meaning
ror_id,str,0,https://ror.org/04dese585,ROR id used to look the institution up
openalex_id,str,0,I59270414,OpenAlex institution id
display_name,str,0,Indian Institute of Science Bangalore,name OpenAlex uses
type,str,0,education,institution type in OpenAlex
country_code,str,0,IN,ISO country code
city,str,0,Bengaluru,city in OpenAlex
works_count_total,int64,0,91690,"all works ever, per OpenAlex"
cited_by_count_total,int64,0,4503170,"all citations ever, per OpenAlex"
h_index,int64,0,468,h-index of the institution's works
two_yr_mean_citedness,float64,0,3.399908592321755,mean citations of works in the last two years


OpenAlex yearly counts (interim/openalex_yearly_*.csv)
  file : interim/openalex_yearly_2026-11-02.csv
  shape: (8100, 11)


,dtype,missing,example,meaning
ror_id,str,0,https://ror.org/04dese585,ROR id
openalex_id,str,0,I59270414,OpenAlex institution id
year,int64,0,1854,publication year
works_all_lineage,int64,0,1,"works that year, institution plus units beneath it"
works_article_lineage,int64,0,1,"of those, type = article"
works_cited_ge1_lineage,int64,0,1,"of those, with at least 1 citation"
works_cited_ge10_lineage,int64,0,0,"of those, with at least 10 citations"
works_all_own,int64,0,1,works that year whose authors list the institution itself
oa_works_count_year,float64,5321,NaN,OpenAlex's own yearly count (starts in 2010)
oa_cited_by_count_year,float64,5321,NaN,"citations received that year, to all works (starts in 2010)"


ROR to OpenAlex crosswalk (interim/openalex_crosswalk_*.csv)
  file : interim/openalex_crosswalk_2026-11-02.csv
  shape: (164, 5)


,dtype,missing,example,meaning
id_new,str,0,U-0220,our institution id
name_new,str,0,indian institute of science,our institution name
decision,str,0,auto,how the ROR match was made: auto or manual
ror_id,str,0,https://ror.org/04dese585,ROR id
openalex_id,str,0,I59270414,OpenAlex institution id
